# 4.5 — Phasor Transformer Direct: Sequence Prediction with the PhasorTransformer Model

This notebook demonstrates sequence prediction using the high-level `PhasorTransformer` class from `PhasorFlow.models`. Unlike notebooks 4.1–4.4 which build the FNet-style circuit from scratch, here we use the **scikit-learn-style API** (`fit`, `predict`, `score`) that wraps all circuit construction internally.

We will:
1. Generate synthetic timeseries data
2. Prepare windowed train/test sequences
3. Train a single-circuit PhasorTransformer
4. Train a stacked PhasorTransformer with threshold gate
5. Use autoregressive generation
6. Visualize all results

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import math
import numpy as np
import torch
import matplotlib.pyplot as plt

import PhasorFlow as pf
from PhasorFlow.models import PhasorTransformer

np.random.seed(42)
torch.manual_seed(42)

## 1. Synthetic Timeseries Data

We generate a composite waveform signal and encode it into the phase domain via $\arcsin$.

In [ ]:
SEQ_LENGTH = 10  # T = number of phasor threads
N_TOTAL = 300

# Composite waveform
t = np.linspace(0, 10 * np.pi, N_TOTAL + 1)
signal = np.sin(t) + 0.4 * np.cos(3 * t)
signal = signal / np.max(np.abs(signal))  # normalize to [-1, 1]
signal = np.clip(signal, -0.99, 0.99)     # clamp for arcsin

# Phase-encode: x -> arcsin(x), maps [-1,1] to [-π/2, π/2]
phase_signal = np.arcsin(signal).astype(np.float32)

# Window into (input_sequence, next_step_target) pairs
X_all, y_all = [], []
for i in range(N_TOTAL - SEQ_LENGTH):
    X_all.append(phase_signal[i:i+SEQ_LENGTH])
    y_all.append(phase_signal[i+SEQ_LENGTH])

X_all = torch.tensor(np.array(X_all))
y_all = torch.tensor(np.array(y_all))

# Train/test split (80/20)
split = int(0.8 * len(X_all))
X_train, y_train = X_all[:split], y_all[:split]
X_test, y_test = X_all[split:], y_all[split:]

print(f'Total samples: {len(X_all)}')
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Sequence length: {SEQ_LENGTH}')

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(signal[:100], label='Raw signal', alpha=0.7)
plt.plot(phase_signal[:100], label='Phase-encoded (arcsin)', alpha=0.7)
plt.xlabel('Time Step'); plt.ylabel('Value')
plt.title('Raw vs Phase-Encoded Signal')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Single-Circuit PhasorTransformer (Notebook 4.1 style)

All FNet blocks in one circuit:

```
[Data Encode] → [Pre-Shift → DFT → Post-Shift] × 2 → arcsin(sin(θ₀))
```

In [ ]:
model_single = PhasorTransformer(
    seq_length=SEQ_LENGTH,
    num_blocks=2,
    stacking='single',
)
print(model_single)
print(f'Total params: {model_single.total_params}')

In [ ]:
result_single = model_single.fit(
    X_train, y_train,
    epochs=30,
    lr=0.05,
    verbose=True,
    print_every=5,
)

In [ ]:
train_mse = model_single.score(X_train, y_train)
test_mse = model_single.score(X_test, y_test)
print(f'Single-circuit | Train MSE: {train_mse:.4f} | Test MSE: {test_mse:.4f}')

## 3. Stacked PhasorTransformer with Threshold Gate (Notebook 4.2 style)

Separate circuits per block with a threshold gate between:

```
[Block 1 circuit] → threshold(τ=0.5) → [Block 2 circuit] → arcsin(sin(θ₀))
```

In [ ]:
model_stacked = PhasorTransformer(
    seq_length=SEQ_LENGTH,
    num_blocks=2,
    stacking='separate',
    inter_block='threshold',
    threshold_tau=0.5,
)
print(model_stacked)
print(f'Total params: {model_stacked.total_params}')

In [ ]:
result_stacked = model_stacked.fit(
    X_train, y_train,
    epochs=30,
    lr=0.05,
    verbose=True,
    print_every=5,
)

In [ ]:
train_mse2 = model_stacked.score(X_train, y_train)
test_mse2 = model_stacked.score(X_test, y_test)
print(f'Stacked+Threshold | Train MSE: {train_mse2:.4f} | Test MSE: {test_mse2:.4f}')

## 4. Training Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(result_single['loss_history'], label='Single-circuit (4.1 style)')
axes[0].plot(result_stacked['loss_history'], label='Stacked+Threshold (4.2 style)')
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Predictions vs actual (test set, first 30 samples)
with torch.no_grad():
    preds_single = model_single.predict(X_test[:30])
    preds_stacked = model_stacked.predict(X_test[:30])

axes[1].plot(y_test[:30].numpy(), 'k-', label='Actual', linewidth=2, alpha=0.8)
axes[1].plot(preds_single.numpy(), '--', label='Single-circuit', alpha=0.7)
axes[1].plot(preds_stacked.numpy(), '--', label='Stacked+Threshold', alpha=0.7)
axes[1].set_title('Predictions vs Actual (Test)'); axes[1].set_xlabel('Sample')
axes[1].set_ylabel('Phase Value'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 5. Autoregressive Generation (Notebook 4.4 / LPM style)

Starting from a context window, the model predicts the next step and rolls the window forward. This is the core of the Large Phasor Model (LPM).

In [ ]:
# Use the stacked model for autoregressive generation
context = X_test[0]  # initial context window
horizon = 40

ar_predictions = model_stacked.predict_autoregressive(context, horizon=horizon)

# Compare with actual future values
actual_future = y_test[:horizon].numpy()
generated = ar_predictions.numpy()

plt.figure(figsize=(12, 5))
plt.plot(context.numpy(), 'b-', linewidth=2, label='Context Window')
x_future = np.arange(SEQ_LENGTH, SEQ_LENGTH + horizon)
plt.plot(x_future, actual_future, 'k-', linewidth=2, label='Actual Future', alpha=0.8)
plt.plot(x_future, generated, 'r--', linewidth=2, label='Autoregressive Generated', alpha=0.8)
plt.axvline(x=SEQ_LENGTH - 0.5, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Time Step'); plt.ylabel('Phase Value')
plt.title('Autoregressive Sequence Generation')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Circuit Introspection

The `get_circuit()` method lets us visualize the underlying phasor circuit.

In [ ]:
# Single-circuit: the full circuit
circuit = model_single.get_circuit(X_test[0])
print(f'Single-circuit mode: {circuit.num_threads} threads, {len(circuit.operations)} operations')
pf.draw(circuit, mode='text')

## 7. Summary

This notebook demonstrated sequence prediction using the `PhasorTransformer` model from `PhasorFlow.models`:

```python
from PhasorFlow.models import PhasorTransformer

# Single-circuit (notebooks 4.1, 4.3)
model = PhasorTransformer(seq_length=10, num_blocks=2)
model.fit(X_train, y_train, epochs=30, lr=0.05)

# Stacked + threshold (notebook 4.2)
model = PhasorTransformer(seq_length=10, num_blocks=2,
                          stacking='separate', inter_block='threshold')

# Autoregressive generation (notebook 4.4 / LPM)
future = model.predict_autoregressive(context, horizon=40)
```

For first-principles implementations, see notebooks **4.1–4.4**.

In [ ]:
print('=== PhasorTransformer Summary ===')
print(f'Seq length:             {SEQ_LENGTH}')
print(f'Single-circuit params:  {model_single.total_params}')
print(f'Stacked params:         {model_stacked.total_params}')
print(f'Single Train MSE:       {train_mse:.4f}')
print(f'Single Test MSE:        {test_mse:.4f}')
print(f'Stacked Train MSE:      {train_mse2:.4f}')
print(f'Stacked Test MSE:       {test_mse2:.4f}')